# Lab 1 — Python Data Structures

**Day 02 · Python for Data Science · Cisco AI/ML Training**

---

## Goals

1. Work with **lists**, **tuples**, **dicts**, and **sets** on restaurant and delivery data.
2. Explain **mutable** vs **immutable** and pick the right structure for the job.
3. Parse rows from the **Zomato CSV** using only core Python (preview before pandas in Lab 3).
4. Deduplicate cuisines and filter records with **set operations**.

> **Checkpoint:** `len(cities) == 4` and the cuisines `set` contains **3** unique values.

**Data:** `data/zomato/zomato_restaurants.csv` (**500** rows) — we sample slices in this lab.

**Day 2 flow:** **Lab 1 (you are here)** — Python collections → Lab 2 NumPy → Lab 3 pandas on Zomato → Labs 4–6 visualization and linear regression.

## Why this matters

Before pandas gives you a DataFrame, data often arrives as **lists of dicts** (API JSON), **column name lists**, or **category sets**. NumPy arrays (Lab 2) and DataFrames (Lab 3) are built on these four types. Getting comfortable here saves debugging time later.

## Quick reference

| Structure | Ordered? | Mutable? | Restaurant example |
|-----------|----------|----------|--------------------|
| `list` | Yes | Yes | Cities in rollout order |
| `tuple` | Yes | No | Fixed `(seed, sample_size)` config |
| `dict` | Yes (3.7+) | Yes | One restaurant JSON record |
| `set` | No | Yes | Unique cuisines in a city |

---

## 0. Locate the Zomato file

In [ ]:
from pathlib import Path
import csv

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

ZOMATO_CSV = GH_ROOT / "data" / "zomato" / "zomato_restaurants.csv"

print("GH root:", GH_ROOT)
print("Zomato CSV exists:", ZOMATO_CSV.is_file())
print("Path:", ZOMATO_CSV.relative_to(GH_ROOT))


### 0b. Column names as a list

CSV files store a **header row**. We read it with the `csv` module — no pandas yet.

In [ ]:
with ZOMATO_CSV.open(newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader)
    first_three_rows = [next(reader) for _ in range(3)]

print("Columns (list):", header)
print("Number of columns:", len(header))
for i, row in enumerate(first_three_rows, start=1):
    print(f"Row {i}: name={row[1]!r}, city={row[2]!r}, rating={row[4]}")


---

## 1. Lists — ordered, mutable sequences

Use a **list** when order matters and you may add or change items — e.g. cities queued for a delivery launch.

In [ ]:
cities = ["Bengaluru", "Mumbai", "Delhi", "Hyderabad"]
print("Type:", type(cities))
print("Length:", len(cities))
print("First:", cities[0], "| Last:", cities[-1])


### 1b. Indexing and slicing

In [ ]:
print("Index 1 (Mumbai):", cities[1])
print("Slice [1:3]:", cities[1:3])       # Mumbai, Delhi — stop index excluded
print("Last two:", cities[-2:])          # negative index counts from the end
print("Copy via slice [:]:", cities[:])  # shallow copy of the whole list


### 1c. Mutating a list

In [ ]:
cities.append("Chennai")
print("After append:", cities)

cities.insert(1, "Pune")  # insert before Mumbai
print("After insert at 1:", cities)

# extend merges another iterable
cities.extend(["Kolkata"])
print("After extend:", cities)


For the **checkpoint** at the end we reset to four cities. Append exercises above are practice — do not skip the reset cell later.

### 1d. Sorting and membership

In [ ]:
rollout = ["Bengaluru", "Mumbai", "Delhi", "Hyderabad"]
rollout_sorted = sorted(rollout)  # sorted() returns new list; original unchanged
print("Alphabetical:", rollout_sorted)

print("'Mumbai' in rollout:", "Mumbai" in rollout)
print("'Chennai' in rollout:", "Chennai" in rollout)

for i, city in enumerate(rollout, start=1):
    print(f"  Launch wave {i}: {city}")


### 1e. Nested list — combo menu

In [ ]:
combo_meal = [
    "Biryani",
    ["Raita", "Salad", "Papad"],
    "Gulab Jamun",
]
print("Main:", combo_meal[0])
print("Sides:", combo_meal[1])
print("First side:", combo_meal[1][0])  # nested indexing


### 1f. Build a city list from real Zomato rows

Read the first **100** restaurants and collect **unique cities** (order of first appearance).

In [ ]:
cities_seen = []
with ZOMATO_CSV.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
        if i >= 100:
            break
        city = row["city"]
        if city not in cities_seen:
            cities_seen.append(city)

print(f"Unique cities in first 100 rows: {len(cities_seen)}")
print("Sample:", cities_seen[:8])


### 1g. List comprehension — city names from file

In [ ]:
with ZOMATO_CSV.open(newline="", encoding="utf-8") as f:
    rows_50 = []
    for i, row in enumerate(csv.DictReader(f)):
        if i >= 50:
            break
        rows_50.append(row)

city_names = [r["city"] for r in rows_50]
print("First 50 rows → list length:", len(city_names))
print("Distinct via set:", len(set(city_names)))


### 1h. `sort` in place vs `sorted` copy

In [ ]:
price_tiers = [299, 149, 499, 199, 349]
copy_sorted = sorted(price_tiers)
print("sorted() returns new list:", copy_sorted)
print("original unchanged:", price_tiers)

price_tiers.sort()
print("after .sort() in place:", price_tiers)


### 1i. Reset checkpoint cities

In [ ]:
cities = ["Bengaluru", "Mumbai", "Delhi", "Hyderabad"]
assert len(cities) == 4
print("Checkpoint cities:", cities)


---

## 2. Tuples — fixed records and config

A **tuple** is ordered but **cannot** be changed after creation. Use it for settings that must not drift mid-notebook.

In [ ]:
ratings = (3.8, 4.2, 3.5, 4.0)
print("Tuple:", ratings)
print("Mean (manual):", round(sum(ratings) / len(ratings), 2))


### 2b. Unpacking

In [ ]:
first, second, third, fourth = ratings
print("Best of four sample ratings:", max(ratings))
name, city, rating = ("Demo Bistro", cities[0], ratings[0])
print(f"{name} in {city} → {rating}")


### 2c. Config tuple for reproducible sampling (used again in Lab 3)

In [ ]:
DATA_CONFIG = (42, 500)  # (random_seed, max_rows)
seed, max_rows = DATA_CONFIG
print(f"Lab config: seed={seed}, cap={max_rows} rows")


### 2d. Immutability — deliberate error

In [ ]:
# ratings[0] = 5.0  # uncomment → TypeError
print("Tuple unchanged:", ratings)


### 2e. Tuple from one Zomato row

In [ ]:
with ZOMATO_CSV.open(newline="", encoding="utf-8") as f:
    row = next(csv.DictReader(f))

snapshot = (row["name"], row["city"], float(row["aggregate_rating"]))
print("Restaurant snapshot tuple:", snapshot)
print("Type of snapshot:", type(snapshot))


---

## 3. Dictionaries — key/value records

APIs and FastAPI handlers exchange **dict**-shaped JSON. Each restaurant is naturally a dict.

In [ ]:
restaurant = {
    "name": "Demo Bistro",
    "city": cities[0],
    "rating": ratings[0],
    "cuisines": {"North Indian", "Chinese"},
    "online_order": True,
    "votes": 1840,
}
for key, value in restaurant.items():
    print(f"  {key}: {value}")


### 3b. Safe access with `.get()`

In [ ]:
print("book_table:", restaurant.get("book_table", False))
print("votes:", restaurant.get("votes", 0))

restaurant["book_table"] = False
restaurant["average_cost_for_two"] = 650
print("After enrichment:", restaurant)


### 3c. Nested dict — city → outlet count (manual tally, first 80 rows)

In [ ]:
outlets_by_city: dict[str, int] = {}
with ZOMATO_CSV.open(newline="", encoding="utf-8") as f:
    for i, row in enumerate(csv.DictReader(f)):
        if i >= 80:
            break
        city = row["city"]
        outlets_by_city[city] = outlets_by_city.get(city, 0) + 1

top_five = sorted(outlets_by_city.items(), key=lambda x: x[1], reverse=True)[:5]
print("Top 5 cities by outlet count (first 80 rows):")
for city, count in top_five:
    print(f"  {city}: {count}")


### 3d. Parse one CSV row into a dict (same shape as Lab 3 will batch-load)

In [ ]:
with ZOMATO_CSV.open(newline="", encoding="utf-8") as f:
    record = next(csv.DictReader(f))

# CSV values are strings — cast numeric fields when needed
record["aggregate_rating"] = float(record["aggregate_rating"])
record["votes"] = int(record["votes"])
record["average_cost_for_two"] = int(record["average_cost_for_two"])

print(record["name"], "|", record["city"], "| rating", record["aggregate_rating"])


### 3e. List of dicts — first 15 restaurants (preview of Lab 3)

In [ ]:
restaurants: list[dict] = []
with ZOMATO_CSV.open(newline="", encoding="utf-8") as f:
    for i, row in enumerate(csv.DictReader(f)):
        if i >= 15:
            break
        restaurants.append({
            "name": row["name"],
            "city": row["city"],
            "rating": float(row["aggregate_rating"]),
            "online_order": row["online_order"] == "Yes",
        })

print(f"Loaded {len(restaurants)} records")
print("First:", restaurants[0])
print("Last:", restaurants[-1])


### 3f. Filter list of dicts — delivery-enabled only

In [ ]:
delivery_on = [r for r in restaurants if r["online_order"]]
print(f"Delivery enabled: {len(delivery_on)} / {len(restaurants)}")
for r in delivery_on[:4]:
    print(f"  {r['name']} ({r['city']}) — {r['rating']}")


### 3g. Add cuisine tag (mutating nested set)

In [ ]:
restaurant["cuisines"].add("Cafe")
print("Cuisines after add:", sorted(restaurant["cuisines"]))


---

## 4. Sets — unique values and overlap

Zomato stores one primary cuisine per row, but menus and filters use **unique tags**. Sets drop duplicates automatically.

In [ ]:
cuisines = restaurant["cuisines"]
print("Type:", type(cuisines))
print("Unique cuisines:", sorted(cuisines))
print("Count:", len(cuisines))


### 4b. Dedup a messy cuisine list

In [ ]:
raw_tags = [
    "North Indian", "Chinese", "North Indian", "Cafe", "Chinese", "Cafe",
]
unique = set(raw_tags)
print("Raw length:", len(raw_tags), "→ unique:", len(unique), sorted(unique))


### 4c. Collect unique cuisines from 150 Zomato rows

In [ ]:
cuisine_set: set[str] = set()
with ZOMATO_CSV.open(newline="", encoding="utf-8") as f:
    for i, row in enumerate(csv.DictReader(f)):
        if i >= 150:
            break
        cuisine_set.add(row["cuisines"])

print(f"Distinct cuisines in first 150 rows: {len(cuisine_set)}")
print("Sample:", sorted(cuisine_set)[:10])


### 4d. Set operations — filter by dietary tags

In [ ]:
menu_tags = {"spicy", "vegan", "North Indian"}
overlap = cuisines & menu_tags
print("Tags matching menu filter:", overlap)

all_tags = cuisines | menu_tags
print("Union (all tags):", all_tags)

only_menu = menu_tags - cuisines
print("Tags on menu but not in restaurant cuisines:", only_menu)


### 4e. Cities with at least one Italian restaurant (first 200 rows)

In [ ]:
italian_cities: set[str] = set()
with ZOMATO_CSV.open(newline="", encoding="utf-8") as f:
    for i, row in enumerate(csv.DictReader(f)):
        if i >= 200:
            break
        if row["cuisines"] == "Italian":
            italian_cities.add(row["city"])

print("Cities with Italian cuisine:", sorted(italian_cities))


### 4f. Membership speed intuition

In [ ]:
big_city_list = city_names * 4  # repeat to simulate longer list
big_city_set = set(big_city_list)

target = "Mumbai"
print("List length:", len(big_city_list), "| Set size:", len(big_city_set))
print("Both support `in` — sets stay fast as data grows (used heavily in ML pipelines).")
print(f"'{target}' in set:", target in big_city_set)


---

## 5. Mutable vs immutable — when to choose

| Scenario | Use | Reason |
|----------|-----|--------|
| Pipeline step list | `list` | Steps get appended |
| Random seed + row cap | `tuple` | Must not change accidentally |
| REST JSON body | `dict` | Keys map to fields |
| Distinct user IDs / cuisines | `set` | Fast membership, no dupes |

---

## 6. Mini scenario — high-rated picks (structures only)

Find restaurant **names** with `aggregate_rating >= 4.0` in the first **200** rows. No pandas — just loops, dicts, and lists.

In [ ]:
high_rated: list[str] = []
with ZOMATO_CSV.open(newline="", encoding="utf-8") as f:
    for i, row in enumerate(csv.DictReader(f)):
        if i >= 200:
            break
        if float(row["aggregate_rating"]) >= 4.0:
            high_rated.append(row["name"])

print(f"Found {len(high_rated)} restaurants rated ≥ 4.0 (first 200 rows)")
print("Examples:", high_rated[:6])


### 6b. Group votes by city (dict of lists)

In [ ]:
votes_by_city: dict[str, list[int]] = {}
with ZOMATO_CSV.open(newline="", encoding="utf-8") as f:
    for i, row in enumerate(csv.DictReader(f)):
        if i >= 120:
            break
        city = row["city"]
        votes_by_city.setdefault(city, []).append(int(row["votes"]))

for city in sorted(votes_by_city)[:4]:
    vlist = votes_by_city[city]
    avg_votes = sum(vlist) / len(vlist)
    print(f"{city}: {len(vlist)} outlets, avg votes ≈ {avg_votes:.0f}")


---

## 7. Final checkpoint

In [ ]:
print("Lab 1 — Python structures")
print(f"cities (list, len={len(cities)}): {cities}")
print(f"ratings (tuple): {ratings}")
print(f"restaurant (dict): name={restaurant['name']}, city={restaurant['city']}")
print(f"unique cuisines (set): {sorted(restaurant['cuisines'])}")

assert len(cities) == 4
assert len(restaurant["cuisines"]) == 3
print("\nNumbers match — you're good.")



---

## 8. Try it yourself (then check below)

1. Build a list of restaurant names where `votes > 3000` (first 100 rows).
2. Store unique `online_order` values (`Yes`/`No`) from those 100 rows in a set.
3. Create a dict `summary` with keys `count` and `avg_rating` for your high-vote list.

In [ ]:
# Your code here (optional — solution in next cell)
high_vote_names: list[str] = []
# ...


In [ ]:
high_vote_names = []
ratings_for_high: list[float] = []
order_flags: set[str] = set()

with ZOMATO_CSV.open(newline="", encoding="utf-8") as f:
    for i, row in enumerate(csv.DictReader(f)):
        if i >= 100:
            break
        order_flags.add(row["online_order"])
        if int(row["votes"]) > 3000:
            high_vote_names.append(row["name"])
            ratings_for_high.append(float(row["aggregate_rating"]))

summary = {
    "count": len(high_vote_names),
    "avg_rating": round(sum(ratings_for_high) / len(ratings_for_high), 2) if ratings_for_high else 0,
}
print("High-vote restaurants:", summary["count"])
print("Average rating:", summary["avg_rating"])
print("Order flags seen:", order_flags)


---

## Reflection

1. Why is a `set` better than a `list` for storing cuisines?
2. Which structure would hold `(random_seed, 500)` for reproducible sampling?
3. Lab 3 loads all **500** rows into a **pandas DataFrame** — which Python type is it closest to?
4. When you used `csv.DictReader`, what Python type was each `row`?

**Next:** [Lab 2 — NumPy arrays](lab02_numpy_arrays.ipynb)